# Loan Approval Model Selection with Grid Search

This notebook uses the same dataset as experiment.ipynb and compares multiple models using GridSearchCV.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import dagshub
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# Same dataset, loaded using a notebook-relative path
data_path = '../data/raw/loan_approval_dataset.csv'
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (4269, 13)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [3]:
target_col = 'loan_status'
drop_cols = [c for c in ['loan_id'] if c in df.columns]

X = df.drop(columns=[target_col] + drop_cols)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print('Numeric columns:', numeric_features)
print('Categorical columns:', categorical_features)

Numeric columns: ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']
Categorical columns: ['education', 'self_employed']


In [8]:
# MLflow + DagsHub setup (same pattern as experiment.ipynb)
mlflow.set_tracking_uri('https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow')
dagshub.init(repo_owner='kevinsangani988', repo_name='Capstone-MLops', mlflow=True)

mlflow.set_experiment('grid search loan_approval_prediction')

Initialized MLflow to track repo "kevinsangani988/Capstone-MLops"

Repository kevinsangani988/Capstone-MLops initialized!

2026/03/14 12:31:28 INFO mlflow.tracking.fluent: Experiment with name 'grid search loan_approval_prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/d371b9fcc7bd4ee28e7d0a69f526b7d1', creation_time=1773487887892, experiment_id='1', last_update_time=1773487887892, lifecycle_stage='active', name='grid search loan_approval_prediction', tags={}>

In [9]:
model_configs = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=500, random_state=42),
        'params': {
            'model__C': [0.1, 1.0, 10.0],
            'model__solver': ['lbfgs', 'liblinear']
        }
    },
    'SVC': {
        'model': SVC(random_state=42),
        'params': {
            'model__C': [0.1, 1.0, 10.0],
            'model__kernel': ['linear', 'rbf'],
            'model__gamma': ['scale', 'auto']
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [None, 10, 20],
            'model__min_samples_split': [2, 5]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [2, 3]
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'model__n_neighbors': [3, 5, 7, 11],
            'model__weights': ['uniform', 'distance'],
            'model__p': [1, 2]
        }
    }
}

In [10]:
results = []
best_estimators = {}

with mlflow.start_run(run_name='GridSearch_Model_Comparison') as parent_run:
    for model_name, cfg in model_configs.items():
        print(f'\nRunning GridSearchCV for: {model_name}')

        pipe = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', cfg['model'])
        ])

        grid = GridSearchCV(
            estimator=pipe,
            param_grid=cfg['params'],
            cv=5,
            scoring='accuracy',
            n_jobs=-1,
            verbose=1
        )

        with mlflow.start_run(run_name=f'{model_name}_grid_search', nested=True):
            grid.fit(X_train, y_train)

            best_estimators[model_name] = grid.best_estimator_

            y_pred = grid.best_estimator_.predict(X_test)
            test_acc = accuracy_score(y_test, y_pred)

            clean_best_params = {
                k.replace('model__', ''): (
                    v if isinstance(v, (int, float, str, bool)) else str(v)
                )
                for k, v in grid.best_params_.items()
            }

            mlflow.log_params(clean_best_params)
            mlflow.log_metric('best_cv_accuracy', float(grid.best_score_))
            mlflow.log_metric('test_accuracy', float(test_acc))

            results.append({
                'model': model_name,
                'best_cv_score': grid.best_score_,
                'test_accuracy': test_acc,
                'best_params': grid.best_params_
            })

    results_df = pd.DataFrame(results).sort_values(by='test_accuracy', ascending=False).reset_index(drop=True)

    best_model_name = results_df.loc[0, 'model']
    best_model = best_estimators[best_model_name]
    best_test_accuracy = float(results_df.loc[0, 'test_accuracy'])

    mlflow.log_param('best_model_name', best_model_name)
    mlflow.log_metric('best_model_test_accuracy', best_test_accuracy)
    mlflow.log_text(results_df.to_csv(index=False), 'grid_search_leaderboard.csv')
    mlflow.sklearn.log_model(best_model, name='best_model')

results_df


Running GridSearchCV for: LogisticRegression
Fitting 5 folds for each of 6 candidates, totalling 30 fits
🏃 View run LogisticRegression_grid_search at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/1/runs/56df9185851a47129a7cae00de84d024
🧪 View experiment at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/1

Running GridSearchCV for: SVC
Fitting 5 folds for each of 12 candidates, totalling 60 fits
🏃 View run SVC_grid_search at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/1/runs/aaf89511f217498ab095f9eb84a9d33e
🧪 View experiment at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/1

Running GridSearchCV for: RandomForest
Fitting 5 folds for each of 12 candidates, totalling 60 fits
🏃 View run RandomForest_grid_search at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/1/runs/cbfc15ffb9d84797aca61a4955cc2e38
🧪 View experiment at: https://dagshub.com/kevinsangani98

,model,best_cv_score,test_accuracy,best_params
0,RandomForest,0.978917,0.982436,"{'model__max_depth': None, 'model__min_samples..."
1,GradientBoosting,0.982723,0.982436,"{'model__learning_rate': 0.1, 'model__max_dept..."
2,SVC,0.945534,0.951991,"{'model__C': 10.0, 'model__gamma': 'auto', 'mo..."
3,LogisticRegression,0.912152,0.922717,"{'model__C': 1.0, 'model__solver': 'lbfgs'}"
4,KNeighbors,0.903075,0.922717,"{'model__n_neighbors': 11, 'model__p': 1, 'mod..."


In [11]:
best_model_name = results_df.loc[0, 'model']
best_model = best_estimators[best_model_name]

print(f'Best model: {best_model_name}')
print('Best params:', results_df.loc[0, 'best_params'])

y_pred_best = best_model.predict(X_test)

print('\nClassification Report:')
print(classification_report(y_test, y_pred_best))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_best))

Best model: RandomForest
Best params: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 200}

Classification Report:
              precision    recall  f1-score   support

    Approved       0.98      0.99      0.99       531
    Rejected       0.99      0.96      0.98       323

    accuracy                           0.98       854
   macro avg       0.98      0.98      0.98       854
weighted avg       0.98      0.98      0.98       854

Confusion Matrix:
[[528   3]
 [ 12 311]]
